# Command center

The workspace for this project: every knob, the setup that acts on them, the features being
built, and the output you check them by. Runs top to bottom on a fresh runtime — no hidden
state, no cell that has to run out of order.

Config comes **first**, before anything acts: read one cell and you know everything this
project can be told to do. Setup then consumes those values. Nothing below reads a setting
from anywhere else.

Features get built *here*, in as many cells as it takes to see them working, and move into
`src/` once they've stopped changing. `AGENTS.md` §6 has that lifecycle; `PLAYBOOK.md` has the
manual setup and the Colab <-> GitHub round trip.

## 1. Config

Every knob, in one cell, before anything runs. Edit here, then run everything below.

In [ ]:
"""Every knob this project has, and why each is set the way it is.

Read this cell on its own and you should know what you can change, what the alternatives were,
and why the current value won — that last part is what gets forgotten first. Grouped by the
decision you're making, not by whichever module consumes the value.

This cell only *declares*. It mounts nothing, creates nothing, downloads nothing — so you can
re-read and re-run it any time without side effects. Setup, below, is what acts on it.
"""

from pathlib import Path

# --- Where files live ---------------------------------------------------------------------
# Everything that isn't code, config or docs: data, weights, caches, outputs, secrets.
# None of it belongs in git — .gitignore keeps it out.
#
# DRIVE_ROOT — alternatives considered:
#   MyDrive/projects/<project>  <- current. survives runtime teardown, and the shared parent
#                                  means MyDrive's root gains one folder however many projects
#                                  you start, rather than one folder per project
#   MyDrive/<project>              a click shallower. fine for one project, clutters the root
#                                  once there are several
#   /content/<anything>            the runtime's own disk. faster, but wiped whenever the
#                                  runtime recycles, which eventually means lost work
#   a folder inside the repo       would put data in git. no
#
# SET ON FIRST SESSION (START_HERE.md): rename the last folder after your project. Two projects
# both left on "NDD" share one Drive folder and quietly overwrite each other's data.
DRIVE_ROOT = Path("/content/drive/MyDrive/projects/NDD")

DATA_DIR = DRIVE_ROOT / "data"          # inputs, placed here by hand (PLAYBOOK.md)
OUTPUTS_DIR = DRIVE_ROOT / "outputs"    # anything a run produces

# --- Where the code comes from ------------------------------------------------------------
# Opening this notebook from GitHub gives you the notebook and nothing else, so the repo is
# fetched separately to put `src/` on the import path. That's what makes `from pipeline import
# ...` work further down.
#
# REPO_DIR — alternatives considered:
#   /content/repo   <- current. fast, and a clone is disposable: it re-clones in seconds
#   under Drive        would survive restarts, but git over Drive's FUSE mount is slow and
#                      occasionally corrupts .git. not worth it for something re-fetchable
#
# REPO_BRANCH — which branch the runtime imports from:
#   main              <- current. what has landed, and what you normally want
#   a feature branch     to check work before it is merged. Open this notebook from that branch
#                        too — the README badge always points at main, so change `blob/main` to
#                        `blob/<branch>` in the URL. Set it back to main afterwards.
#
# SET ON FIRST SESSION (START_HERE.md): point REPO_URL at your own repo.
REPO_URL = "https://github.com/vak-sah/NDD-notebook-driven-development.git"
REPO_BRANCH = "main"
REPO_DIR = Path("/content/repo")

# --- Dependencies -------------------------------------------------------------------------
# Packages Colab doesn't already ship. Pin versions — a runtime six months from now should
# resolve to what this one did. Empty until the project needs something.
PACKAGES: list[str] = [
    # "some-lib==1.2.3",
]

# --- Project knobs ------------------------------------------------------------------------
# Empty until this project has features of its own. Each arrives as its own block: the value,
# the alternatives weighed, and one line on why the current value won.

print(f"DRIVE_ROOT   {DRIVE_ROOT}")
print(f"DATA_DIR     {DATA_DIR}")
print(f"OUTPUTS_DIR  {OUTPUTS_DIR}")
print(f"REPO_URL     {REPO_URL} ({REPO_BRANCH})")
print(f"PACKAGES     {PACKAGES or 'none'}")

## 2. Setup

Acts on the config above. Run once per fresh runtime; re-run after **Runtime > Restart**.

In [ ]:
"""Make the runtime match the config above: mount Drive, fetch the repo, install packages.

There are no settings in this cell by design — every value it uses comes from the config cell,
so there is exactly one place to look when changing how a run behaves. If you find yourself
about to hardcode something here, it belongs up there instead.

Idempotent, and safe to re-run after changing REPO_BRANCH: an existing clone is moved onto the
requested branch rather than re-cloned, and the Drive folders are created only if missing.
"""

import subprocess
import sys

from google.colab import drive

drive.mount("/content/drive")

for _d in (DATA_DIR, OUTPUTS_DIR):
    _d.mkdir(parents=True, exist_ok=True)

if (REPO_DIR / ".git").exists():
    # fetch + checkout + reset rather than a bare `pull`: pull would update whatever branch
    # happened to be checked out, so changing REPO_BRANCH between runs would silently do
    # nothing. The clone is disposable and nobody edits it, so a hard reset is safe and makes
    # the runtime match the branch exactly.
    git = ["git", "-C", str(REPO_DIR)]
    subprocess.run([*git, "fetch", "--quiet", "origin", REPO_BRANCH], check=True)
    subprocess.run([*git, "checkout", "--quiet", REPO_BRANCH], check=True)
    subprocess.run([*git, "reset", "--hard", "-q", f"origin/{REPO_BRANCH}"], check=True)
else:
    subprocess.run(
        ["git", "clone", "--quiet", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)],
        check=True,
    )

if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))

# START_HERE.md is deleted the moment onboarding completes, so finding it in the clone means
# this repo was never made specific — and REPO_URL above is probably still the template's.
# Without this the cells below run someone else's code and look perfectly healthy doing it.
if (REPO_DIR / "START_HERE.md").exists():
    print(
        "\n!!  This repo has not been onboarded: START_HERE.md is still in it.\n"
        f"    REPO_URL is {REPO_URL}\n"
        "    If that is not this repo, everything below is running the template's code,\n"
        "    not yours, and any output is meaningless.\n"
        "    Fix: open an agent session on this repo and say hello. If onboarding already\n"
        "    ran, its changes never reached the default branch — check for an unmerged\n"
        "    branch or an unpushed commit.\n"
    )

if PACKAGES:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *PACKAGES], check=True)

print(f"Drive mounted \u00b7 {REPO_DIR} on {REPO_BRANCH} \u00b7 {len(PACKAGES)} extra package(s)")

## 3. Pipeline

The end-to-end stub: read a file, pass the records through untouched, write them out. It proves
notebook -> `src/` -> `tests/` -> CI is connected, and gives the first real feature something to
replace. `src/pipeline/stub.py` owns it — delete it once a real stage exists.

In [ ]:
"""Run the stub end to end and show the result.

Writes a small input file into DATA_DIR, runs the pipeline, prints what came back out. Replace
this call when the stub is replaced by a real first stage.
"""

from pipeline import stub

in_path = DATA_DIR / "stub_input.txt"
out_path = OUTPUTS_DIR / "stub_output.txt"

in_path.write_text("alpha\nbeta\ngamma\n")

written = stub.run(in_path, out_path)

print(f"{written} record(s) -> {out_path}\n")
print(out_path.read_text())